In [1]:
import pandas as pd
import arcpy

In [2]:
import os

ENVIRONMENT = r'F:\GIS\PROJECTS\Transportation\Protect\PROTECT_analysis\PROTECT_analysis.gdb'
streets_network = r'F:\GIS\PROJECTS\Transportation\Protect\PROTECT_analysis\PROTECT_analysis.gdb\Streets_Network_Drive'
oid_field = "OBJECTID"  # OID field name for streets_network
block_group_layer = r'F:\GIS\PROJECTS\Transportation\Protect\PROTECT_analysis\PROTECT_analysis.gdb\Block_Group_Layer'


In [ ]:
# list fields from block_group_layer
fields = arcpy.ListFields(block_group_layer)
for field in fields:
    print(f"'{field.name}'")


In [ ]:
def assign_max_polygon_values(line_fc, polygon_fc, line_oid_field, polygon_value_fields, output_fields=None):
    """
    Assign the maximum value from intersecting polygons to each line segment.

    Args:
        line_fc: Path to line feature class
        polygon_fc: Path to polygon feature class
        line_oid_field: OID field name in line_fc (e.g., 'OBJECTID')
        polygon_value_fields: List of field names to get max values from (e.g., ['criticality', 'exposure'])
        output_fields: List of output field names (if None, uses polygon_value_fields + '_max')

    Returns:
        Dictionary mapping line OID to dict of {output_field: max_value}
    """
    if output_fields is None:
        output_fields = [f"{field}_max" for field in polygon_value_fields]

    # Create output fields if they don't exist
    for out_field in output_fields:
        try:
            arcpy.management.AddField(line_fc, out_field, "DOUBLE")
        except:
            pass  # Field may already exist

    results = {}
    
    # Load all polygons into memory first (as a list, not dict)
    polygon_data = []
    with arcpy.da.SearchCursor(polygon_fc, ["SHAPE@"] + polygon_value_fields) as cursor:
        for row in cursor:
            poly_geom = row[0]
            values = {polygon_value_fields[i]: row[i + 1] for i in range(len(polygon_value_fields))}
            polygon_data.append((poly_geom, values))

    # Get line geometries and OIDs
    with arcpy.da.SearchCursor(line_fc, [line_oid_field, "SHAPE@"]) as line_cursor:
        for line_oid, line_geom in line_cursor:
            max_values = {field: None for field in output_fields}

            # Check intersection with each polygon
            for poly_geom, poly_values in polygon_data:
                try:
                    # Check if geometries are NOT disjoint (meaning they intersect in some way)
                    if not line_geom.disjoint(poly_geom):
                        # Get max value for each field
                        for i, field in enumerate(polygon_value_fields):
                            value = poly_values.get(field)
                            if value is not None:
                                if max_values[output_fields[i]] is None:
                                    max_values[output_fields[i]] = value
                                else:
                                    max_values[output_fields[i]] = max(max_values[output_fields[i]], value)
                except:
                    pass  # Skip if geometry operation fails

            results[line_oid] = max_values

    # Update the line feature class with the max values
    with arcpy.da.UpdateCursor(line_fc, [line_oid_field] + output_fields) as update_cursor:
        for row in update_cursor:
            line_oid = row[0]
            if line_oid in results:
                for i, field in enumerate(output_fields):
                    row[i + 1] = results[line_oid][field]
                update_cursor.updateRow(row)

    return results


# Example usage:
# results = assign_max_polygon_values(
#     line_fc=streets_network,
#     polygon_fc=block_group_layer,
#     line_oid_field=oid_field,
#     polygon_value_fields=['criticality', 'exposure'],
#     output_fields=['criticality_max', 'exposure_max']
# )

In [9]:
def assign_max_buffered_line_values(line_fc, buffer_line_fc, line_oid_field, buffer_value_fields, buffer_distance, output_fields=None):
    """
    Assign the maximum value from buffered line segments to each line in the primary feature class.

    Args:
        line_fc: Path to primary line feature class
        buffer_line_fc: Path to line feature class to buffer
        line_oid_field: OID field name in line_fc (e.g., 'OBJECTID')
        buffer_value_fields: List of field names to get max values from (e.g., ['criticality', 'exposure'])
        buffer_distance: Buffer distance in feature class units (e.g., 100 for 100 meters)
        output_fields: List of output field names (if None, uses buffer_value_fields + '_max')

    Returns:
        Dictionary mapping line OID to dict of {output_field: max_value}
    """
    if output_fields is None:
        output_fields = [f"{field}_max" for field in buffer_value_fields]

    # Get existing fields
    existing_fields = [f.name for f in arcpy.ListFields(line_fc)]
    
    # Create output fields if they don't exist
    for out_field in output_fields:
        if out_field not in existing_fields:
            try:
                arcpy.management.AddField(line_fc, out_field, "DOUBLE")
                print(f"Created field: {out_field}")
            except Exception as e:
                print(f"Warning: Could not create field {out_field}: {e}")

    results = {}
    
    # Load all buffer lines and create buffers
    buffer_data = []
    with arcpy.da.SearchCursor(buffer_line_fc, ["SHAPE@"] + buffer_value_fields) as cursor:
        for row in cursor:
            line_geom = row[0]
            buffered_geom = line_geom.buffer(buffer_distance)
            values = {buffer_value_fields[i]: row[i + 1] for i in range(len(buffer_value_fields))}
            buffer_data.append((buffered_geom, values))

    print(f"Loaded {len(buffer_data)} buffer line(s)")

    # Get primary line geometries and OIDs
    with arcpy.da.SearchCursor(line_fc, [line_oid_field, "SHAPE@"]) as line_cursor:
        for line_oid, line_geom in line_cursor:
            max_values = {field: None for field in output_fields}

            # Check intersection with each buffered line
            if line_geom is not None:
                shp = _wkb.loads(bytes(line_geom.WKB))
                hits = [i for i in tree.query(shp) if shp.intersects(buffer_data[i][0])]
                for idx in hits:
                        for i, field in enumerate(buffer_value_fields):
                            value = buffer_values.get(field)
                            if value is not None:
                                if max_values[output_fields[i]] is None:
                                    max_values[output_fields[i]] = value
                                else:
                                    max_values[output_fields[i]] = max(max_values[output_fields[i]], value)

            results[line_oid] = max_values

    # Update the line feature class with the max values
    with arcpy.da.UpdateCursor(line_fc, [line_oid_field] + output_fields) as update_cursor:
        for row in update_cursor:
            line_oid = row[0]
            if line_oid in results:
                for i, field in enumerate(output_fields):
                    row[i + 1] = results[line_oid][field]
                update_cursor.updateRow(row)

    print(f"Updated {len(results)} line segment(s)")
    return results


# Example usage:
# results = assign_max_buffered_line_values(
#     line_fc=streets_network,
#     buffer_line_fc=transit_lines,
#     line_oid_field=oid_field,
#     buffer_value_fields=['accessibility', 'frequency'],
#     buffer_distance=100,  # 100 meter buffer
#     output_fields=['transit_accessibility_max', 'transit_frequency_max']
# )

In [13]:
def assign_max_line_join_values(line_fc, join_line_fc, line_oid_field, join_value_fields,
                                output_fields=None, buffer_distance=5,
                                intersection_threshold=0.5, force_recreate_fields=False):
    """
    Spatial join with intersection length weighting: for each line in line_fc,
    find the join line whose buffered geometry has the most overlap, filtered by
    a minimum intersection threshold.

    Args:
        line_fc: Path to primary line feature class (target)
        join_line_fc: Path to line feature class to join from (source)
        line_oid_field: OID field name in line_fc
        join_value_fields: Field names to read from join_line_fc
        output_fields: Output field names (defaults to join_value_fields + '_max')
        buffer_distance: Buffer distance in feature class units (metres for UTM).
            Defaults to 5 metres. Used to handle slightly-offset parallel segments.
        intersection_threshold: Minimum overlap as fraction of target line length.
            Defaults to 0.5 (50%). A target line is only populated if its overlap
            with the buffered join line is >= this fraction of its total length.
        force_recreate_fields: Delete and recreate wrongly-typed output fields
            that already hold data.

    Returns:
        Dictionary mapping line OID to dict of {output_field: value}
    """
    import arcpy
    from shapely import wkb as _wkb
    from shapely.strtree import STRtree

    if output_fields is None:
        output_fields = [f"{field}_max" for field in join_value_fields]
    if len(output_fields) != len(join_value_fields):
        raise ValueError("output_fields and join_value_fields must be the same length")

    ADD_TYPE = {
        "SmallInteger": "SHORT", "Integer": "LONG", "BigInteger": "BIGINTEGER",
        "Single": "FLOAT", "Double": "DOUBLE", "String": "TEXT",
        "Date": "DATE", "OID": "LONG", "GUID": "GUID", "GlobalID": "GUID",
    }
    NUMERIC = {"SmallInteger", "Integer", "BigInteger", "Single", "Double", "OID"}
    COMPATIBLE = {
        "SHORT": {"SmallInteger", "Integer", "BigInteger", "Double", "Single"},
        "LONG": {"Integer", "BigInteger", "Double"},
        "BIGINTEGER": {"BigInteger", "Double"},
        "FLOAT": {"Single", "Double"},
        "DOUBLE": {"Double"},
        "TEXT": {"String"},
        "DATE": {"Date"},
        "GUID": {"GUID", "GlobalID", "String"},
    }

    # ------------------------------------------------------------------
    # Validate up front
    # ------------------------------------------------------------------
    join_flds = {f.name: f for f in arcpy.ListFields(join_line_fc)}
    missing = [f for f in join_value_fields if f not in join_flds]
    if missing:
        raise ValueError(f"{missing} not found in {join_line_fc}")

    line_flds = {f.name: f for f in arcpy.ListFields(line_fc)}
    for src, out in zip(join_value_fields, output_fields):
        src_type = join_flds[src].type
        need = ADD_TYPE.get(src_type, "DOUBLE")
        kwargs = {}
        if need == "TEXT":
            kwargs["field_length"] = max(join_flds[src].length or 255, 255)

        existing = line_flds.get(out)
        if existing is None:
            arcpy.management.AddField(line_fc, out, need, **kwargs)
            print(f"Created field: {out} ({need}, from {src} {src_type})")
            continue

        if existing.type in COMPATIBLE.get(need, set()):
            continue

        populated = 0
        with arcpy.da.SearchCursor(line_fc, [out]) as _c:
            for (_v,) in _c:
                if _v is not None:
                    populated += 1
                    break
        if populated and not force_recreate_fields:
            raise ValueError(
                f"Field '{out}' is {existing.type} but '{src}' needs {need}, and the "
                f"field already holds data. Pass force_recreate_fields=True to drop "
                f"and rebuild it, or choose a different output field name."
            )
        arcpy.management.DeleteField(line_fc, out)
        arcpy.management.AddField(line_fc, out, need, **kwargs)
        print(f"Recreated field: {out} ({existing.type} -> {need})")

    # ------------------------------------------------------------------
    # Load join geometries, reproject if needed, buffer them
    # ------------------------------------------------------------------
    target_sr = arcpy.Describe(line_fc).spatialReference
    join_sr = arcpy.Describe(join_line_fc).spatialReference
    reproject = (target_sr.factoryCode or -1) != (join_sr.factoryCode or -2)
    if reproject:
        print(f"Reprojecting join lines: {join_sr.name} -> {target_sr.name}")

    print(f"Buffering join lines by {buffer_distance} units")
    join_shapes, join_values = [], []
    with arcpy.da.SearchCursor(join_line_fc, ["SHAPE@"] + join_value_fields) as cursor:
        for row in cursor:
            if row[0] is None:
                continue
            geom = row[0].projectAs(target_sr) if reproject else row[0]
            # Convert to shapely, buffer it (polygon), store the values
            shp = _wkb.loads(bytes(geom.WKB))
            buffered = shp.buffer(buffer_distance)
            join_shapes.append(buffered)
            join_values.append({f: row[i + 1] for i, f in enumerate(join_value_fields)})

    print(f"Loaded {len(join_shapes)} buffered join line(s)")
    tree = STRtree(join_shapes)

    # ------------------------------------------------------------------
    # For each target line: find the buffered join geometry with the longest
    # intersection, but only if that intersection is >= threshold fraction
    # of the target line's length
    # ------------------------------------------------------------------
    results = {}
    n_matched = 0
    with arcpy.da.SearchCursor(line_fc, [line_oid_field, "SHAPE@"]) as line_cursor:
        for line_oid, line_geom in line_cursor:
            vals = {f: None for f in output_fields}

            if line_geom is not None:
                target_line = _wkb.loads(bytes(line_geom.WKB))
                target_length = target_line.length
                min_overlap = intersection_threshold * target_length

                # Find candidate buffered polygons by bounding box
                candidates = tree.query(target_line)
                best_overlap_length = 0
                best_source_idx = None

                for idx in candidates:
                    # Calculate actual intersection
                    intersection = target_line.intersection(join_shapes[idx])
                    if intersection.is_empty:
                        continue

                    # For a line-polygon intersection, get the total overlap length
                    overlap_length = intersection.length
                    if overlap_length >= min_overlap and overlap_length > best_overlap_length:
                        best_overlap_length = overlap_length
                        best_source_idx = idx

                # Populate from the best match
                if best_source_idx is not None:
                    n_matched += 1
                    for src, out in zip(join_value_fields, output_fields):
                        vals[out] = join_values[best_source_idx][src]

            results[line_oid] = vals

    print(f"{n_matched:,} of {len(results):,} line(s) matched a buffered join line "
          f"(with {int(intersection_threshold*100)}%+ overlap)")

    # ------------------------------------------------------------------
    # Update output fields
    # ------------------------------------------------------------------
    with arcpy.da.UpdateCursor(line_fc, [line_oid_field] + output_fields) as update_cursor:
        for row in update_cursor:
            if row[0] in results:
                vals = results[row[0]]
                for i, field in enumerate(output_fields):
                    row[i + 1] = vals[field]
                update_cursor.updateRow(row)

    print(f"Updated {len(results)} line segment(s)")
    return results


# Example usage:
# results = assign_max_line_join_values(
#     line_fc=streets_network,
#     join_line_fc=aadt_feature_class,
#     line_oid_field=oid_field,
#     join_value_fields=['Estimated_2024_AADT', 'name'],
#     output_fields=['Estimated_2024_AADT', 'aadt_zone_name'],
#     buffer_distance=5,              # 5 metre buffer to catch offset segments
#     intersection_threshold=0.5,     # Street must overlap buffered AADT by >= 50%
# )

In [4]:
disadvantaged_fields = ['Below_Poverty_Household',
                        'Speak_English_Not_Well_Not_at_A',
                        'Vehicle_Available_0',
                        'With_Disability']

In [ ]:
results = assign_max_polygon_values(
     line_fc=streets_network,
     polygon_fc=block_group_layer,
     line_oid_field=oid_field,
      polygon_value_fields=disadvantaged_fields,
      output_fields=[f"{field}_max" for field in disadvantaged_fields]
)

In [5]:
slr_fc = r'F:\GIS\PROJECTS\Transportation\Protect\PROTECT_analysis\PROTECT_analysis.gdb\tahoe_slr'


In [ ]:
results_line = assign_max_buffered_line_values(
    line_fc=streets_network,
    buffer_line_fc=slr_fc,
    line_oid_field=oid_field,
    buffer_value_fields=['diff_lengt', 'detour'],
    buffer_distance=10,  # 100 meter buffer
    output_fields=['diff_length_max', 'detour_max']
)

In [8]:
results = assign_max_line_join_values(
    line_fc=streets_network,
    join_line_fc=slr_fc,
    line_oid_field=oid_field,
    join_value_fields=['diff_lengt', 'detour'],
    output_fields=['slr_diff_length_max', 'slr_detour_max']
)

Loaded 74647 join line(s)
Updated 95285 line segment(s)


In [11]:
aadt_feature_class = r'F:\GIS\PROJECTS\Transportation\Protect\PROTECT_analysis\PROTECT_analysis.gdb\AADT_Streetlight_Attributed'
aadt_fields = ['Estimated_2024_AADT', 'name']
aadt_output_fields = ['Estimated_2024_AADT', 'aadt_zone_name']


In [14]:
results = assign_max_line_join_values(
    line_fc=streets_network,
    join_line_fc=aadt_feature_class,
    line_oid_field=oid_field,
    join_value_fields=aadt_fields,
    output_fields=aadt_output_fields,
    buffer_distance=5,              # 5m buffer to catch offset parallel segments
    intersection_threshold=0.5,     # Only populate if >=50% of street overlaps buffered AADT
)

Reprojecting join lines: GCS_WGS_1984 -> NAD_1983_UTM_Zone_10N
Buffering join lines by 5 units
Loaded 1319 buffered join line(s)
7,856 of 95,285 line(s) matched a buffered join line (with 50%+ overlap)
Updated 95285 line segment(s)


In [ ]:
def assign_max_neighboring_line_values(line_fc, line_oid_field, filter_where_clause, value_field):
    """
    For each line segment matching a filter, return the maximum value of a field
    from all adjoining (touching) line segments.

    Args:
        line_fc: Path to line feature class
        line_oid_field: OID field name (e.g., 'OBJECTID')
        filter_where_clause: SQL WHERE clause to filter target segments 
                           (e.g., "slr_detour_max = 0 AND class IN ('primary', 'secondary', 'tertiary', 'trunk')")
        value_field: Field name to read max value from neighbors (e.g., 'slr_diff_length_max')

    Returns:
        Dictionary mapping line OID to max_value from neighbors
    """
    from shapely import wkb as _wkb
    
    # Load all line geometries and the value field
    all_lines = {}  # oid -> (geometry, value)
    with arcpy.da.SearchCursor(line_fc, [line_oid_field, "SHAPE@", value_field]) as cursor:
        for oid, geom, value in cursor:
            if geom is not None:
                all_lines[oid] = (geom, value)
    
    print(f"Loaded {len(all_lines)} line geometries")
    
    # Get only the target lines that match the filter
    target_oids = set()
    with arcpy.da.SearchCursor(line_fc, [line_oid_field], where_clause=filter_where_clause) as cursor:
        for (oid,) in cursor:
            target_oids.add(oid)
    
    print(f"Found {len(target_oids)} line(s) matching filter: {filter_where_clause}")
    
    results = {}
    
    # For each target line, find neighbors and get max value
    for target_oid in target_oids:
        if target_oid not in all_lines:
            results[target_oid] = None
            continue
        
        target_geom, _ = all_lines[target_oid]
        max_neighbor_value = None
        
        # Check all other lines for adjacency
        for neighbor_oid, (neighbor_geom, neighbor_value) in all_lines.items():
            if neighbor_oid == target_oid or neighbor_value is None:
                continue
            
            # Check if geometries touch (adjacent)
            try:
                if target_geom.touches(neighbor_geom):
                    if max_neighbor_value is None:
                        max_neighbor_value = neighbor_value
                    else:
                        max_neighbor_value = max(max_neighbor_value, neighbor_value)
            except:
                pass  # Skip if geometry operation fails
        
        results[target_oid] = max_neighbor_value
    
    return results

In [19]:
streets_network = r'F:\GIS\PROJECTS\Transportation\Protect\PROTECT_analysis\PROTECT_analysis.gdb\Streets_Network_Tahoe'
results = assign_max_neighboring_line_values(
    line_fc=streets_network,
    line_oid_field='OBJECTID',
    filter_where_clause="slr_detour_max = 0 AND class IN ('primary', 'secondary', 'tertiary', 'trunk')",
    value_field='slr_diff_length_max',
    output_field='slr_diff_length_max_from_neighbors'
)

Loaded 18285 line geometries
Found 340 line(s) matching filter: slr_detour_max = 0 AND class IN ('primary', 'secondary', 'tertiary', 'trunk')
Updated 340 line(s); 338 populated with neighbor values


In [20]:
streets_network = r'F:\GIS\PROJECTS\Transportation\Protect\PROTECT_analysis\PROTECT_analysis.gdb\Streets_Network_Tahoe'
results = assign_max_neighboring_line_values(
    line_fc=streets_network,
    line_oid_field='OBJECTID',
    filter_where_clause="slr_diff_length_max IS NULL AND class IN ('primary', 'secondary', 'tertiary', 'trunk')",
    value_field='slr_diff_length_max',
    output_field='slr_diff_length_max_from_neighbors'
)

Loaded 18285 line geometries
Found 1115 line(s) matching filter: slr_diff_length_max IS NULL AND class IN ('primary', 'secondary', 'tertiary', 'trunk')
Updated 1115 line(s); 1105 populated with neighbor values
